In [1]:
%cd ../..

/home/olarinoyem/Research/hospitalization_research


In [2]:
import pandas as pd
import numpy as np
import os

import shutil
import joblib
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error as mse
from src.forecasting.ml_forecasting import calculate_metrics
from src.utils import ts_utils
import plotly.express as px
import plotly.graph_objects as go
from itertools import cycle
import time
import warnings
from tqdm.notebook import tqdm
from pathlib import Path

from src.utils import plotting_utils
from src.dl.dataloaders import TimeSeriesDataModule
from src.dl.models import SingleStepRNNConfig, SingleStepRNNModel
import pytorch_lightning as pl
import torch
# For reproduceability set a random seed
pl.seed_everything(42)
tqdm.pandas()
pio.templates.default = "plotly_white"

/home/olarinoyem/Research/hospitalization_research/src/utils/data_utils.py:6: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Global seed set to 42


In [3]:
os.makedirs("data/output", exist_ok=True)
preprocessed = Path("data/NHS_region/Timeseries")
output = Path("data/output")

In [4]:
def format_plot(
    fig, legends=None, xlabel="Time", ylabel="Value", title="", font_size=15
):
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name=next(names)))
    fig.update_layout(
        autosize=False,
        width=900,
        height=500,
        title_text=title,
        title={"x": 0.5, "xanchor": "center", "yanchor": "top"},
        titlefont={"size": 20},
        legend_title=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.98,
            xanchor="right",
            x=1,
        ),
        yaxis=dict(
            title_text=ylabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title_text=xlabel,
            titlefont=dict(size=font_size),
            tickfont=dict(size=font_size),
        ),
    )
    return fig

In [5]:
try:
    train_df = pd.read_csv(preprocessed / "featured_eng_train.csv")
    test_df = pd.read_csv(preprocessed / "featured_eng_test.csv")
    val_df = pd.read_csv(preprocessed / "featured_eng_val.csv")

except FileNotFoundError:
    print("File not found, please run the feature engineering notebook first")

In [6]:
# Filter data for London
# Filter data for London
sample_train_df = train_df.loc[
    train_df.areaName == "London",
    [
        "date",
        'covidOccupiedMVBeds', 'cumAdmissions',
       'hospitalCases', 'newAdmissions', 'covidOccupiedMVBeds_trend',
       'cumAdmissions_trend', 'hospitalCases_trend', 'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
       'covidOccupiedMVBeds_trend_seasonal_diff', 'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]
sample_test_df = test_df.loc[
    test_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]

sample_val_df = val_df.loc[
    val_df.areaName == "London",
    [
        'date',
        'cumAdmissions',
        'hospitalCases', 
        'newAdmissions', 
       'cumAdmissions_trend',
       'hospitalCases_trend',
       'newAdmissions_trend',
       'covidOccupiedMVBeds_trend_diff',
    'covidOccupiedMVBeds_lag_1',
       'covidOccupiedMVBeds_lag_7', 'covidOccupiedMVBeds_lag_14',
       'covidOccupiedMVBeds_lag_21', 'covidOccupiedMVBeds_rolling_7_mean',
       'covidOccupiedMVBeds_rolling_7_std', 'day_of_week', 'week_of_year',
       'Month', 'Quarter'
    ],
]

In [7]:
# Convert date to datetime and set as index
sample_train_df["date"] = pd.to_datetime(sample_train_df["date"])
sample_test_df["date"] = pd.to_datetime(sample_test_df["date"])
sample_val_df["date"] = pd.to_datetime(sample_val_df["date"])

sample_train_df.set_index("date", inplace=True)
sample_test_df.set_index("date", inplace=True)
sample_val_df.set_index("date", inplace=True)

In [8]:
sample_train_df.head()

,covidOccupiedMVBeds,cumAdmissions,hospitalCases,newAdmissions,covidOccupiedMVBeds_trend,cumAdmissions_trend,hospitalCases_trend,newAdmissions_trend,covidOccupiedMVBeds_trend_diff,covidOccupiedMVBeds_trend_seasonal_diff,covidOccupiedMVBeds_lag_1,covidOccupiedMVBeds_lag_7,covidOccupiedMVBeds_lag_14,covidOccupiedMVBeds_lag_21,covidOccupiedMVBeds_rolling_7_mean,covidOccupiedMVBeds_rolling_7_std,day_of_week,week_of_year,Month,Quarter
date,,,,,,,,,,,,,,,,,,,,
2022-06-13,61,123251,1001,114,61.000000,123595.142857,1003.142857,114.857143,-1.142857,-16.000000,57.0,69.0,75.0,73.0,61.000000,4.472136,0,24,6,2
2022-06-12,60,123137,967,88,60.000000,123480.285714,991.714286,109.428571,-1.000000,-14.571429,61.0,67.0,84.0,74.0,60.000000,3.605551,6,23,6,2
2022-06-11,60,123049,942,83,59.000000,123370.857143,981.571429,106.285714,-1.000000,-13.571429,60.0,67.0,81.0,72.0,59.000000,1.914854,5,23,6,2
2022-06-10,54,122966,932,93,58.571429,123264.571429,971.714286,104.285714,-0.428571,-10.285714,60.0,57.0,83.0,70.0,58.571429,2.636737,4,23,6,2
2022-06-09,51,122873,914,76,57.714286,123160.285714,960.142857,98.714286,-0.857143,-8.428571,54.0,57.0,76.0,70.0,57.714286,3.903600,3,23,6,2


In [9]:
target = "covidOccupiedMVBeds_trend_diff"
index_cols = ["date", "areaName"]
pred_df = pd.concat([sample_train_df[[target]], sample_test_df[[target]]])

In [10]:
sample_train_df['type'] = "train"
sample_val_df['type'] = "val"
sample_test_df['type'] = "test"
sample_df = pd.concat([sample_train_df[[target, "type"]], sample_val_df[[target, "type"]], sample_test_df[[target, "type"]],])
sample_df['covidOccupiedMVBeds_trend_diff'] = sample_df['covidOccupiedMVBeds_trend_diff'].astype('float32')
sample_df.head()

,covidOccupiedMVBeds_trend_diff,type
date,,
2022-06-13,-1.142857,train
2022-06-12,-1.000000,train
2022-06-11,-1.000000,train
2022-06-10,-0.428571,train
2022-06-09,-0.857143,train


In [11]:
# # Assuming sample_train_df, sample_val_df, and sample_test_df are your DataFrames
# features = [col for col in sample_train_df.columns if col not in ["date", "covidOccupiedMVBeds_trend", "type"]]
# train_data = sample_train_df[features]
# val_data = sample_val_df[features]
# test_data = sample_test_df[features]


In [12]:
# full_data = pd.concat([train_data, val_data, test_data])
# full_data = full_data.astype({col: 'float32' for col in full_data.columns})


In [13]:
# datamodule = TimeSeriesDataModule(
#         data=full_data.values, # pass the values as a NumPy array
#         n_val=val_data.shape[0],
#         n_test=test_data.shape[0],
#         window=7,  # 7 days window
#         horizon=1,  # single step
#         normalize="global",  # normalizing the data
#         batch_size=32,
#         num_workers=0
# )
# datamodule.setup()


In [14]:
datamodule = TimeSeriesDataModule(data = sample_df[[target]],
        n_val = sample_val_df.shape[0],
        n_test = sample_test_df.shape[0],
        window = 7, # 7 days window  
        horizon = 1, # single step
        normalize = "global", # normalizing the data
        batch_size = 32,
        num_workers = 0)
datamodule.setup()

In [15]:
rnn_config = SingleStepRNNConfig(
    rnn_type="RNN",
    input_size=1,   # 1 for univariate time series
    hidden_size=64,
    num_layers=3,
    bidirectional=False,
    learning_rate=1e-3
)
model = SingleStepRNNModel(rnn_config)
model.float()

SingleStepRNNModel(
  (rnn): RNN(1, 64, num_layers=3, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (loss): MSELoss()
)

In [16]:
# Getting a batch from the train_dataloader
for batch in datamodule.train_dataloader():
    x, y = batch
    break
print("Shape of x: ",x.shape)
print("Shape of y: ",y.shape)

Shape of x:  torch.Size([32, 7, 1])
Shape of y:  torch.Size([32, 1, 1])


In [17]:
# Getting a batch from the train_dataloader
for batch in datamodule.train_dataloader():
    x, y = batch
    break
print("Data type of x: ", x.dtype) # Should print torch.float32
print("Data type of y: ", y.dtype) # Should print torch.float32


Data type of x:  torch.float32
Data type of y:  torch.float32


In [18]:
for param in model.parameters():
    print(param.dtype) # Should print torch.float32


torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32
torch.float32


In [19]:
x.float(), y.float()

(tensor([[[ 1.4994e+00],
          [ 1.7748e+00],
          [ 1.4994e+00],
          [ 2.1878e+00],
          [ 2.0845e+00],
          [ 2.8417e+00],
          [ 2.7901e+00]],
 
         [[-4.9326e-02],
          [ 7.1134e-02],
          [ 1.9508e-02],
          [-1.4909e-02],
          [ 1.9508e-02],
          [-8.3743e-02],
          [-2.0420e-01]],
 
         [[ 2.4322e-01],
          [ 2.4322e-01],
          [ 3.4647e-01],
          [ 4.6693e-01],
          [ 3.9810e-01],
          [ 4.1530e-01],
          [ 3.1205e-01]],
 
         [[ 1.2929e+00],
          [ 1.3790e+00],
          [ 1.3790e+00],
          [ 1.0348e+00],
          [ 1.1725e+00],
          [ 1.6887e+00],
          [ 1.8436e+00]],
 
         [[ 1.2276e-01],
          [ 5.3925e-02],
          [ 1.9508e-02],
          [ 1.0555e-01],
          [ 1.9508e-02],
          [ 1.2276e-01],
          [ 8.8342e-02]],
 
         [[ 2.2996e-03],
          [ 1.9508e-02],
          [ 1.3997e-01],
          [ 1.9159e-01],
          

In [20]:
batch = (batch[0].float(), batch[1].float())  # Convert the batch data to float precision
y_hat, y = model(batch)
print("Shape of y_hat: ",y_hat.shape)
print("Shape of y: ",y.shape)

Shape of y_hat:  torch.Size([32, 7, 1])
Shape of y:  torch.Size([32, 7, 1])


In [21]:
# Calculating the loss
l = model.loss(y_hat, y)
print(l)

tensor(0.9856, grad_fn=<MseLossBackward0>)


In [22]:
trainer = pl.Trainer(
    min_epochs=5,
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=3)],
)
trainer.fit(model, datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
[rank: 0] Global seed set to 42
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name | Type    | Params
---------------------------------
0 | rnn  | RNN     | 20.9 K
1 | fc   | Linear  | 65    
2 | loss | MSELoss | 0     
---------------------------------
21.0 K    Trainable params
0         Non-trainable params
21.0 K    Total params
0.084     Total estimated model pa

Sanity Checking: 0it [00:00, ?it/s]

/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:442: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 32 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  2.38it/s]

/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: PossibleUserWarning: It is recommended to use `self.log('valid_loss', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
  warning_cache.warn(
/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: PossibleUserWarning: It is recommended to use `self.log('valid_MAE', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
  warning_cache.warn(


/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:442: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 32 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/loops/fit_loop.py:281: PossibleUserWarning: The number of training batches (13) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Epoch 0: 100%|██████████| 13/13 [00:01<00:00, 12.83it/s, v_num=2, train_loss=0.302, valid_loss=0.374, valid_MAE=0.354]

/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: PossibleUserWarning: It is recommended to use `self.log('train_MAE', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
  warning_cache.warn(


Epoch 8: 100%|██████████| 13/13 [00:00<00:00, 99.76it/s, v_num=2, train_loss=0.378, valid_loss=0.164, valid_MAE=0.177, train_MAE=0.173]  


In [23]:
# Removing artifacts created during training
shutil.rmtree("lightning_logs")

In [24]:
def mase(actual, predicted, insample_actual):
    mae_insample = np.mean(np.abs(np.diff(insample_actual)))
    mae_outsample = np.mean(np.abs(actual - predicted))
    return mae_outsample / mae_insample


def forecast_bias(actual, predicted):
    return np.mean(predicted - actual)


In [25]:
def plot_forecast(pred_df, forecast_columns, forecast_display_names=None, save_path=None):
    if forecast_display_names is None:
        forecast_display_names = forecast_columns
    else:
        assert len(forecast_columns) == len(forecast_display_names)

    mask = ~pred_df[forecast_columns[0]].isnull()
    colors = [
        "rgba(" + ",".join([str(c) for c in plotting_utils.hex_to_rgb(c)]) + ",<alpha>)"
        for c in px.colors.qualitative.Plotly
    ]
    act_color = colors[0]
    colors = cycle(colors[1:])
    
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=pred_df[mask].index,
            y=pred_df[mask].covidOccupiedMVBeds_trend_diff,
            mode="lines",
            line=dict(color=act_color.replace("<alpha>", "0.9")),
            name="7 day-moving average MVBeds_trends",
        )
    )
    for col, display_col in zip(forecast_columns, forecast_display_names):
        fig.add_trace(
            go.Scatter(
                x=pred_df[mask].index,
                y=pred_df.loc[mask, col],
                mode="lines",
                line=dict(dash="dot", color=next(colors).replace("<alpha>", "1")),
                name=display_col,
            )
        )
    
    return fig


def highlight_abs_min(s, props=""):
    return np.where(s == np.nanmin(np.abs(s.values)), props, "")


In [26]:
metric_record = []

In [27]:
pred = trainer.predict(model, datamodule.test_dataloader())
pred = torch.cat(pred).squeeze().detach().numpy()
pred = pred * datamodule.train.std + datamodule.train.mean
actuals = sample_test_df[target].values
algorithm_name = rnn_config.rnn_type

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)


[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
Missing logger folder: /home/olarinoyem/Research/hospitalization_research/lightning_logs
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

Missing logger folder: /home/olarinoyem/Research/hospitalization_research/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]
/home/olarinoyem/miniconda3/envs/deep_tf/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:442: PossibleUserWarning: The dataloader, predict_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value

Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 20.63it/s]


TypeError: cat(): argument 'tensors' (position 1) must be tuple of Tensors, not NoneType

In [ ]:
pred_df_ = pd.DataFrame({"Vanilla RNN": pred}, index=sample_test_df.index)
pred_df = sample_test_df.join(pred_df_)

In [ ]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=["Vanilla RNN"], forecast_display_names=["Vanilla RNN"])
title = f"{rnn_config.rnn_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

## LSTM


In [ ]:
rnn_config = SingleStepRNNConfig(
    rnn_type="LSTM",
    input_size=1,
    hidden_size=128,
    num_layers=2,
    bidirectional=False,
    learning_rate=1e-3,
)

model = SingleStepRNNModel(rnn_config)

trainer = pl.Trainer(
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=3)],
)
trainer.fit(model, datamodule)


# Removing artifacts created during training
shutil.rmtree("lightning_logs")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type    | Params
---------------------------------
0 | rnn  | LSTM    | 199 K 
1 | fc   | Linear  | 129   
2 | loss | MSELoss | 0     
---------------------------------
199 K     Trainable params
0         Non-trainable params
199 K     Total params
0.797     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/loops/fit_loop.py:280: PossibleUserWarning:

The number of training batches (25) is smaller than the loggi

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
pred = trainer.predict(model, datamodule.test_dataloader())
pred = torch.cat(pred).squeeze().detach().numpy()
pred = pred * datamodule.train.std + datamodule.train.mean
actuals = sample_test_df[target].values
algorithm_name = rnn_config.rnn_type

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, predict_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.



Predicting: 0it [00:00, ?it/s]

{'Algorithm': 'LSTM', 'MAE': '0.7912', 'MSE': '1.2090', 'MASE': '0.5518', 'Forecast Bias': '-0.16'}


In [ ]:
pred_df_ = pd.DataFrame({"LSTM": pred}, index=sample_test_df.index)
pred_df = sample_test_df.join(pred_df_)

In [ ]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=["LSTM"], forecast_display_names=["LSTM"])
title = f"{rnn_config.rnn_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

## GRU


In [ ]:
rnn_config = SingleStepRNNConfig(
    rnn_type="GRU",
    input_size=1,
    hidden_size=64,
    num_layers=2,
    bidirectional=False,
    learning_rate=1e-3,
)

model = SingleStepRNNModel(rnn_config)

trainer = pl.Trainer(
    max_epochs=100,
    callbacks=[pl.callbacks.EarlyStopping(monitor="valid_loss", patience=3)],
)
trainer.fit(model, datamodule)
# Removing artifacts created during training
shutil.rmtree("lightning_logs")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type    | Params
---------------------------------
0 | rnn  | GRU     | 37.8 K
1 | fc   | Linear  | 65    
2 | loss | MSELoss | 0     
---------------------------------
37.9 K    Trainable params
0         Non-trainable params
37.9 K    Total params
0.152     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.

/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/loops/fit_loop.py:280: PossibleUserWarning:

The number of training batches (25) is smaller than the loggi

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [ ]:
pred = trainer.predict(model, datamodule.test_dataloader())
pred = torch.cat(pred).squeeze().detach().numpy()
pred = pred * datamodule.train.std + datamodule.train.mean
actuals = sample_test_df[target].values
algorithm_name = rnn_config.rnn_type

metrics = {
    "Algorithm": algorithm_name,
    "MAE": mae(actuals, pred),
    "MSE": mse(actuals, pred),
    "MASE": mase(actuals, pred, sample_train_df[target].values),
    "Forecast Bias": forecast_bias(actuals, pred),
}

value_formats = ["{}", "{:.4f}", "{:.4f}", "{:.4f}", "{:.2f}"] # Added a format for the Algorithm name
metrics = {key: format_.format(value) for key, value, format_ in zip(metrics.keys(), metrics.values(), value_formats)}
metric_record.append(metrics)
print(metrics)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/michaelajao/miniconda3/envs/deep_tf/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning:

The dataloader, predict_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.



Predicting: 0it [00:00, ?it/s]

{'Algorithm': 'GRU', 'MAE': '0.8008', 'MSE': '1.2164', 'MASE': '0.5585', 'Forecast Bias': '0.04'}


In [ ]:
pred_df_ = pd.DataFrame({"GRU": pred}, index=sample_test_df.index)
pred_df = sample_test_df.join(pred_df_)

In [ ]:
# Plotting the forecast
fig = plot_forecast(pred_df, forecast_columns=["GRU"], forecast_display_names=["GRU"])
title = f"{rnn_config.rnn_type}: MAE: {metrics['MAE']} | MSE: {metrics['MSE']} | MASE: {metrics['MASE']} | Bias: {metrics['Forecast Bias']}"
fig = format_plot(fig, title=title)
fig.update_xaxes(type="date", range=["2022-03-01", "2022-05-01"])
fig.show()

In [ ]:
metric_record

[{'Algorithm': 'RNN',
  'MAE': '0.8637',
  'MSE': '1.2894',
  'MASE': '0.6023',
  'Forecast Bias': '0.37'},
 {'Algorithm': 'LSTM',
  'MAE': '0.7912',
  'MSE': '1.2090',
  'MASE': '0.5518',
  'Forecast Bias': '-0.16'},
 {'Algorithm': 'GRU',
  'MAE': '0.8008',
  'MSE': '1.2164',
  'MASE': '0.5585',
  'Forecast Bias': '0.04'}]

In [ ]:
metric_df = pd.DataFrame(metric_record)
metric_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Algorithm      3 non-null      object
 1   MAE            3 non-null      object
 2   MSE            3 non-null      object
 3   MASE           3 non-null      object
 4   Forecast Bias  3 non-null      object
dtypes: object(5)
memory usage: 248.0+ bytes


In [ ]:
metric_df[["MAE", "MSE", "MASE", "Forecast Bias"]] = metric_df[["MAE", "MSE", "MASE", "Forecast Bias"]].astype('float32')


In [ ]:
formatted = metric_df.style.format({
    "MAE": "{:.4f}",
    "MSE": "{:.4f}",
    "MASE": "{:.4f}",
    "Forecast Bias": "{:.2f}%",
    "Time Elapsed": "{:.6f}",
})
formatted = formatted.highlight_min(
    color="lightgreen", subset=["MAE", "MSE", "MASE"]
).apply(
    highlight_abs_min,
    props="color:black;background-color:lightgreen",
    axis=0,
    subset=["Forecast Bias"],
)
formatted


,Algorithm,MAE,MSE,MASE,Forecast Bias
0,RNN,0.8637,1.2894,0.6023,0.37%
1,LSTM,0.7912,1.2090,0.5518,-0.16%
2,GRU,0.8008,1.2164,0.5585,0.04%
